In [1]:
import sys
import os

# 1. Add the external folder to Python's search path
sys.path.append(os.path.abspath(".."))

In [3]:
from guardrails import Guard
from langchain_chroma import Chroma
from pathlib import Path
from dotenv import load_dotenv
from src.config.parameters_config import params_config
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from guardrails_ai.provenance_embeddings import ProvenanceEmbeddings
# from guardrails_ai.reading_time import ReadingTime
from guardrails_ai.relevancy_evaluator import RelevancyEvaluator
import numpy as np

c:\Users\Jay Kanakia\Desktop\CampusX\llmops-rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [6]:
app_params = params_config.rag_app

In [13]:
REPO_ROOT = Path.cwd().parent
VECTOR_STORE_DIR = REPO_ROOT / "saved_embeddings"

In [14]:
embedder = OpenAIEmbeddings(model=app_params.embedder_model,
                            dimensions=app_params.embedder_dimensions) # 1536
chunk_size = app_params.chunk_size
chunk_overlap = app_params.chunk_overlap

In [15]:
def load_knowledge_base():
    # vector store
    vs = Chroma(collection_name=app_params.collection_name,
            embedding_function=embedder,
            persist_directory=VECTOR_STORE_DIR.as_posix())


    return vs
if VECTOR_STORE_DIR.exists():
    vs = load_knowledge_base()

In [17]:
def get_retriever():
    # create the retriever
    retriever = vs.as_retriever(search_type=app_params.search_type,
                                search_kwargs={"k":app_params.k})

    
    return retriever

retriever = get_retriever()

In [18]:
def embed_func(texts):
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    if isinstance(texts, str):
        return np.array(embedding_model.embed_query(texts))

    return np.array(embedding_model.embed_documents(texts))

In [19]:
# generate the guardrails

# reading_time = ReadingTime(
#     reading_time=3,
#     on_fail="noop"
# )

relevancy_eval = RelevancyEvaluator(
    llm_callable="gpt-5-mini",
    on_fail="refrain"
)

hallucination_check = ProvenanceEmbeddings(
    threshold=0.7,
    validation_method="sentence",
    on_fail="refrain",
)

In [20]:
# create the guard

guard = Guard().use(
    # reading_time,
    relevancy_eval,
    hallucination_check
)

In [21]:
query = "What is the diffence between online vs offline evals"
retriever_output = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in retriever_output])

In [22]:
llm = ChatOpenAI(model=app_params.llm)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the user query
                  based on the given context only. If you do not know the answer
                  say I don't know. Do not add any preamble to the response"""),
    ("human", "context: {context}\n\nquery: {query}")
])

rag_chain = prompt | llm | StrOutputParser()
response = rag_chain.invoke({"context": context, "query": query})
context

"And that is why, who comes in the picture?\nOnline eval.\nI can understand the work from the name.\nOnline eval is evaluating your system,\nOn live production traffic,\nAfter deployment,\nAs real users interact with it.\nThe simplest definition of online eval is,\nThat it is a different type of evaluation,\nWhich we run on production,\nAfter our software is deployed.\nAnd the biggest characteristic of it is,\nThat it works without an answer.\nIt works without a golden data set.\nThis is the biggest feature of online eval.\nAnd that is why it is super critical,\nBecause it helps us,\nThat our software, which is deployed,\nKeeps running properly.\nIt tells us that,\nThere is no problem with online.\nIn that sense, online evals are super important.\nSo, on the basis of the discussion till now,\nIf I quickly lay out a difference,\nBetween offline and online,\nThere are 4-5 pointers.\nFirst of all, offline eval,\nIs before deployment,\nAfter online eval deployment.\nTalking about data,\nIn

In [23]:
print(response)

Offline eval
- When: before deployment.
- Data: fixed/golden dataset with an answer key.
- Purpose: gating (CI), version comparison, catch regressions.
- Inputs: only anticipated cases included.
- Cost/speed: fast, cheap, repeatable (small dataset).

Online eval
- When: after deployment, runs on live production traffic.
- Data: no golden dataset or answer key; works without ground truth.
- Purpose: drift detection, monitor real-world chatbot performance, catch surprises/emergent bugs.
- Inputs: unpredictable — anything can occur in production.
- Cost/speed: can be large-scale and costly; sampling is often used to reduce cost.

Summary: they are complementary (not rivals) — offline for pre-deploy gating and regression checks, online for continuous real-world monitoring and drift/emergent-issue detection.


In [24]:
context = """Offline evaluation happens before deployment, using a fixed golden dataset that you
create ahead of time. It's fast, cheap, and repeatable. Online evaluation happens
after deployment, on live production traffic, without a golden dataset."""


query = "What is the difference between offline and online evals?"

response = """MLflow was created by Databricks in 2018 and its enterprise tier costs $499/month,
which includes unlimited experiment tracking and AI copilot features."""

In [ ]:
validation_metadata = {
    "sources": [context],
    "original_prompt": query,
    "embed_function": embed_func,
    "chunk_size": 2,
    "chunk_overlap": 1
}
try:
    outcome = guard.validate(response, metadata=validation_metadata)
    if outcome.validated_output is None:
        print("Please ask again. No relevant response generated")
    if outcome.validation_passed:
        print(f"Response: {outcome.validated_output}")
except Exception as e:
    print(e) 

c:\Users\Jay Kanakia\Desktop\CampusX\llmops-rag\.venv\lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Please ask again. No relevant response generated


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.
